# Healthcare Data Mapping Assistant
## Streamlit implementation

**Student Name:** Shelia Howard  
**Course:** DSC670-T301 Advanced Uses of Generative AI (2271-1)  
**Assignment:** Term Project - Milestone 4  
**Due Date:** 11/22/2026  
**Dataset(s):** `demo_cases.json` (two authored synthetic metadata examples, bundled with the app). Live requests use synthetic user-entered metadata. Historical Milestone 3 results come from the completed report and final Colab run: `mapping_train.csv` (192), `mapping_validation.csv` (48), `mapping_test.csv` (48), `mapping_challenge.csv` (8), and `mapping_legacy.csv` (30). The application does not read these historical CSVs.

### Introduction
I developed a Streamlit interface that suggests a destination in a controlled 13-field demographic schema, explains its recommendation, and asks an analyst to decide what to do with it. I loaded the model I fine-tuned in Milestone 3 and ran two synthetic examples through the same inference and validation functions used by my Streamlit app. This notebook checks the same Python module the app imports. It walks through prompt construction, validating a candidate and a review response, and handling invalid output. The final section loads the saved model checkpoint and verifies live inference on two synthetic examples.

Keep this notebook in the repository’s `notebooks` folder or beside `app.py`. The first code cell locates the application source from either location.

## 1. Locate the Streamlit source

In [1]:
from pathlib import Path
import json
import os
import sys

# Locate the application from either the repository root or notebooks folder.
app_root = Path.cwd().resolve()
if not (app_root / "mapping.py").is_file():
    app_root = app_root.parent

assert (app_root / "mapping.py").is_file(), "Open this notebook within the project repository."

sys.path.insert(0, str(app_root))

from mapping import (
    INPUTS, TARGETS, checkpoint_ready, generate_mapping, load_model,
    messages, read_examples, validate_response,
)

examples = read_examples()
print(f"{len(TARGETS)} approved targets and {len(examples)} illustrative inputs.")

13 approved targets and 2 illustrative inputs.


I moved reusable code into a Python module so the notebook and app share the same prompt and validation rules. The Streamlit entry point manages the form, result display, and analyst decision.

## 2. Construct an inference request

In [2]:
sample = examples[0]["source"]
# Compare only model inputs to ensure reference answers do not leak.
prompt = messages(sample)
assert tuple(json.loads(prompt[1]["content"])) == INPUTS
print(prompt[1]["content"])

{"source_field": "patientDOB", "datatype": "date", "description": "Patient date of birth", "example_value": "1990-04-11"}


The system instruction identifies the allowed targets and escalation rules. The user message contains only four source attributes. In live mode, Qwen's tokenizer renders both messages with the model's chat template.

## 3. Inspect the two analyst workflows

In [3]:
for example in examples:
    # These responses are authored fixtures used to exercise the interface.
    response, valid = validate_response(json.dumps(example["response"]))
    assert valid
    print(example["label"], response["target_field"], response["human_review"])

Date of birth date_of_birth False
Full legal name REVIEW True


The date-of-birth example displays a candidate for an analyst to check. A full legal name needs review because the schema stores name components individually. The interface keeps the analyst decision separate from the recommendation and includes both in the JSON download.

## 4. Check the response guard

In [4]:
cases = {
    "fenced JSON": '```json {"target_field":"city","human_review":false,"reason":"x"} ```',
    "invented target": '{"target_field":"fhir_unknown","human_review":false,"reason":"x"}',
    "contradictory review": '{"target_field":"REVIEW","human_review":false,"reason":"x"}',
    "repeated key": '{"target_field":"city","target_field":"state","human_review":false,"reason":"x"}',
}
for label, raw in cases.items():
    # The guard keeps an invalid answer from appearing as a candidate.
    response, valid = validate_response(raw)
    assert not valid and response["target_field"] == "REVIEW"
    print(label, "-> REVIEW")

fenced JSON -> REVIEW
invented target -> REVIEW
contradictory review -> REVIEW
repeated key -> REVIEW


The app checks approved vocabulary, exact keys, a boolean review flag, and a reason. A valid response can still recommend the wrong destination. The analyst interface addresses that risk by allowing the user to hold, reject, or revise a suggestion.

## 5. Verify live inference with the fine-tuned model

I used `Qwen2.5-0.5B-Instruct` as the base model for fine-tuning in Milestone 3. In this section, I load the saved checkpoint and test it on two synthetic examples using the same inference and validation functions as the Streamlit app.

In [5]:
RUN_LIVE = True
model_dir = Path(os.environ.get("MAPPING_MODEL_DIR", app_root / "final_model"))

if RUN_LIVE:
    # Load only the exported checkpoint to preserve model provenance.
    model, tokenizer = load_model(model_dir)
    for example in examples:
        raw = generate_mapping(model, tokenizer, example["source"])
        response, valid = validate_response(raw)
        print(example["label"], "contract valid:", valid, "response:", response)
else:
    print(f"Checkpoint files detected: {checkpoint_ready(model_dir)}")
    print("Live inference skipped. Follow the README and set RUN_LIVE = True to run it.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Date of birth contract valid: True response: {'target_field': 'date_of_birth', 'human_review': False, 'reason': 'The description identifies date of birth.'}
Full legal name contract valid: True response: {'target_field': 'REVIEW', 'human_review': True, 'reason': 'The metadata is ambiguous, unsupported, contradictory, or needs transformation; analyst review is required.'}


## Conclusion

I loaded my fine-tuned Qwen checkpoint from Milestone 3 and ran live inference on two synthetic examples using the same functions the Streamlit app imports. For `patientDOB`, the model returned `date_of_birth` with `human_review=False` and explained that the description identified a date of birth. For `full_name`, the model returned `REVIEW` with `human_review=True` and gave a general explanation covering ambiguity or a possible need for transformation. In this case, analyst review is appropriate because the source contains a full name in one field and the target schema stores name components separately. Both responses matched the expected decisions and returned `contract valid=True`.

I also verified that the response validator routed four deliberately invalid responses to `REVIEW`. This check matters because the app needs to handle malformed outputs before showing a candidate mapping. A response that passes validation still needs to be checked for meaning, which is why the workflow gives the analyst the final decision.

These two live checks verified checkpoint loading, response generation, and validation in the notebook. The Milestone 3 benchmark provides the broader performance evidence, with 41 of 48 correct targets and 48 of 48 responses meeting the JSON contract on the main test set. Taken together, the milestones connect the course’s work on fine-tuning and structured generation to a practical application where an analyst can review a model’s recommendation before acting on it.

**Source notes.** Howard (2026), *Healthcare Data Mapping Assistant* [DSC670 Milestone 1 proposal, Milestone 3 final report, and completed Colab notebook], Bellevue University. Streamlit Community Cloud deployment steps are linked in the README.